In [ ]:
# import sys
# !{sys.executable} -m pip install earthengine-api geemap numpy matplotlib pandas scikit-learn

# Google Earth Engine - Simple Setup
Easy authentication and basic usage of Google Earth Engine.

In [ ]:
# Step 1: Import libraries
import ee
import os
import geemap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Step 2: Authenticate (this will open a browser)
print("🔐 Authenticating with Google Earth Engine...")
ee.Authenticate()

# Step 3: Initialize
print("🚀 Initializing Google Earth Engine...")
ee.Initialize()

In [ ]:
# Define Austin area
austin = ee.Geometry.Rectangle([-98.1, 30.1, -97.6, 30.5], None, False)

# Load AlphaEarth data
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
# img = (embeddings
#        .filterDate(ee.Date.fromYMD(2023, 1, 1), ee.Date.fromYMD(2024, 1, 1))
#        .filterBounds(austin)
#        .mosaic()
#        .clip(austin))

# Create map
MapAlpha = geemap.Map(center=[30.2672, -97.7431], zoom=10)

# # Add AlphaEarth as RGB (bands A01, A16, A09)
# Map.addLayer(
#     img.select(['A01','A16','A09']),
#     {'min': -0.3, 'max': 0.3},
#     'AlphaEarth RGB'
# )

# Add Austin boundary
MapAlpha.addLayer(austin, {'color': 'blue'}, 'Austin')

# Show the map
MapAlpha

In [ ]:
# Load AlphaEarth embeddings for Austin AOI from 2017 to 2024, one by one
years = range(2017, 2025)
for year in years:
    img_year = (embeddings
        .filterDate(ee.Date.fromYMD(year, 1, 1), ee.Date.fromYMD(year, 12, 31))
        .filterBounds(austin)
        .mosaic()
        .clip(austin))
    MapAlpha.addLayer(
        img_year.select(['A01','A16','A09']),
        {'min': -0.3, 'max': 0.3},
        f'AlphaEarth RGB {year}'
    )
MapAlpha

In [ ]:
# Calculate cosine similarity between consecutive yearly embeddings and show as single-band layers

def normalize(img, band_names):
    norm = img.select(band_names).pow(2).reduce('sum').sqrt()
    return img.select(band_names).divide(norm)

years = list(range(2017, 2025))
images = []
for year in years:
    img_year = (embeddings
        .filterDate(ee.Date.fromYMD(year, 1, 1), ee.Date.fromYMD(year, 12, 31))
        .filterBounds(austin)
        .mosaic()
        .clip(austin))
    images.append(img_year)

MapAlpha2 = geemap.Map(center=[30.2672, -97.7431], zoom=10)
MapAlpha2.addLayer(austin, {'color': 'blue'}, 'Austin')
# Get band names from the first image (all years have same bands)
band_names = images[0].bandNames()

# Normalize images
images_norm = [normalize(img, band_names) for img in images]
for i in range(len(images_norm)-1):
    # Calculate cosine similarity for all bands between year i and i+1
    cos_img = images_norm[i].multiply(images_norm[i+1]).reduce('sum').rename('cosine_similarity')
    cos_img = cos_img.clip(austin)
    MapAlpha2.addLayer(cos_img, {
        'min': 0.5, 'max': 1,
        'palette': ['blue', 'white', 'red']
    }, f'Cosine Similarity {years[i]}-{years[i+1]}')
MapAlpha2

## Change Detection

Here's what the three layers represent and why the previous code was slow.

- Year of Lowest Similarity: For each pixel, we compare consecutive annual embeddings and find the year pair with the lowest cosine similarity (i.e., the biggest spectral/feature change). We report the ending year of that pair. Example: if 2020–2021 is the lowest-similarity pair, the layer shows 2021 at that pixel.
- Magnitude of Changes: The strength of the detected change, computed as 1 − min(cosine_similarity). Values closer to 1 indicate larger changes, while values near 0 indicate stability.
- Duration of Changes: The longest consecutive run of years where change exceeded a threshold (here, >20% change, i.e., 1 − similarity > 0.2). This highlights areas with persistent change over multiple years vs. short blips.

Why it was slow and threw an error:
- The loop used getInfo(), which pulls server data to the client for each image, causing large network transfers and breaking server-side typing (leading to the EEException when trying to rebuild ee.Image from a client-side dict).
- The fix keeps all operations server-side: we map over ImageCollections, use qualityMosaic to pick the year of minimum similarity via an inverted weight, and compute the longest run with a pure server-side iterate. This avoids client round-trips and is much faster and robust.

Tip: You can adjust the change sensitivity by tweaking the threshold value in the duration calculation.

In [ ]:
# # Change detection layers based on cosine similarity (server-side, fast)

# # Create a new map for change detection
# Map_change = geemap.Map(center=[30.2672, -97.7431], zoom=10)

# # Build a server-side ImageCollection of cosine similarity between consecutive years
# # and keep the ending year as a property on each image for later use
# cosine_ic = ee.ImageCollection.fromImages([
#     images_norm[i]
#         .multiply(images_norm[i+1])
#         .reduce(ee.Reducer.sum())
#         .rename('cosine_similarity')
#         .set('pair_end_year', years[i+1])
#         .clip(austin)
#     for i in range(len(images_norm)-1)
# ])

# # Layer 1: Year of lowest similarity
# # 1a) Minimum similarity value per pixel across the collection
# min_similarity_image = cosine_ic.min().rename('lowest_similarity')

# # 1b) Use qualityMosaic on inverted similarity (1 - similarity) so that the image
# #     with the lowest similarity is selected, then take its year band.

# def to_year_inv(img):
#     img = ee.Image(img)
#     end_year = ee.Number(img.get('pair_end_year'))
#     inv = ee.Image(1).subtract(img).rename('inv')  # higher => more change
#     year_band = ee.Image.constant(end_year).toFloat().rename('year')
#     return year_band.addBands(inv)

# year_inv_ic = cosine_ic.map(to_year_inv)
# selected = year_inv_ic.qualityMosaic('inv')
# year_of_lowest_similarity = selected.select('year').rename('year_of_lowest_similarity')

# # Layer 2: Magnitude of changes = 1 - min similarity (same as selected 'inv')
# magnitude_of_changes = selected.select('inv').rename('magnitude_of_changes')

# # Layer 3: Duration of the longest consecutive run of years where change > threshold
# # Define change as (1 - similarity) and threshold as 0.2 (i.e., >20% change)
# threshold = 0.2
# inv_sorted = cosine_ic.sort('pair_end_year').map(
#     lambda img: ee.Image(1).subtract(ee.Image(img)).rename('inv')
#         .set('pair_end_year', img.get('pair_end_year'))
#         .clip(austin)
# )
# mask_ic = inv_sorted.map(
#     lambda img: ee.Image(img).gt(threshold).rename('mask')
#         .set('pair_end_year', img.get('pair_end_year'))
# )

# # Compute longest consecutive run using a server-side iterate (no getInfo)
# mask_list = mask_ic.toList(mask_ic.size())
# init = (ee.Image.constant(0).toInt16().rename('curr')
#         .addBands(ee.Image.constant(0).toInt16().rename('best'))
#         .clip(austin))

# def iter_fn(acc, img):
#     acc = ee.Image(acc)
#     # Ensure we use a single-band 0/1 mask and align band names during math
#     b = ee.Image(img).select('mask').toInt16()
#     curr = acc.select('curr')
#     best = acc.select('best')
#     # If b==1 then increment curr, else reset to 0. Use masking to avoid band name mismatch.
#     inc = curr.add(1)
#     new_curr = inc.updateMask(b).rename('curr').unmask(0)
#     new_best = best.max(new_curr)
#     return new_curr.addBands(new_best.rename('best'))

# result = ee.Image(ee.List(mask_list).iterate(iter_fn, init))
# duration_of_changes = result.select('best').rename('duration')

# # Combine layers into a single image
# change_detection_image = (year_of_lowest_similarity
#                           .addBands(magnitude_of_changes)
#                           .addBands(duration_of_changes))

# # Add layers to the map
# Map_change.addLayer(change_detection_image.select('year_of_lowest_similarity'), {
#     'min': 2017, 'max': 2024,
#     'palette': ['#2166ac','#4393c3','#92c5de','#fddbc7','#d6604d','#b2182b','#67001f']
# }, 'Year of Lowest Similarity')

# Map_change.addLayer(change_detection_image.select('magnitude_of_changes'), {
#     'min': 0, 'max': 1,
#     'palette': ['white','#fee08b','#f46d43','#a50026']
# }, 'Magnitude of Changes')

# Map_change.addLayer(change_detection_image.select('duration'), {
#     'min': 0, 'max': 5,
#     'palette': ['#ffffbf','#fdae61','#f46d43','#d73027','#a50026']
# }, 'Duration of Changes')

# Map_change

In [ ]:
import ee, geemap

# Create a new map for change detection
MapAlpha_change = geemap.Map(center=[30.2672, -97.7431], zoom=10)

# 0) Helper: normalize embeddings to unit L2 per pixel
def normalize(img, band_names, eps=1e-6):
    norm = img.select(band_names).pow(2).reduce(ee.Reducer.sum()).sqrt().add(eps)
    return img.select(band_names).divide(norm)

# 1) Normalize all yearly images (server side)
band_names = images[0].bandNames()
images_norm = [normalize(img, band_names) for img in images]

# Build cosine-similarity ImageCollection for consecutive pairs, and tag each with its pair end-year.
pair_years_py = years[1:]                 # e.g., [2018, 2019, ..., 2024]
pair_years = ee.List(pair_years_py)       # server-side list of integers

cosine_ic = ee.ImageCollection.fromImages([
    images_norm[i]
        .multiply(images_norm[i+1])
        .reduce(ee.Reducer.sum())
        .rename('cosine_similarity')
        .set('pair_end_year', years[i+1])
        .clip(austin)
    for i in range(len(images_norm)-1)
])

cos_list = []
for i in range(len(images_norm) - 1):
    imgA = ee.Image(images_norm[i])
    imgB = ee.Image(images_norm[i + 1])
    cos  = imgA.multiply(imgB).reduce(ee.Reducer.sum()).rename('cos')  # cosine similarity in [0,1]
    yr   = ee.Number(pair_years.get(i))
    cos_with_year = cos.addBands(ee.Image.constant(yr).rename('pair_year').toInt16())
    cos_list.append(cos_with_year.clip(austin))

cos_ic = ee.ImageCollection.fromImages(cos_list)  # bands per image: ['cos', 'pair_year']

# 2) Year of lowest similarity & magnitude
# Use qualityMosaic with a score = (1 - cos), so the max score corresponds to min similarity.
def add_score(img):
    return img.addBands(ee.Image(1).subtract(img.select('cos')).rename('score'))

scored = cos_ic.map(add_score)
picked = scored.qualityMosaic('score')  # picks the image with highest 'score' pixelwise

lowest_similarity      = picked.select('cos').rename('lowest_similarity')
year_of_lowest_similarity = picked.select('pair_year').rename('year_of_lowest_similarity')
magnitude_of_changes   = ee.Image(1).subtract(lowest_similarity).rename('magnitude_of_changes')

# ...existing code...

# Layer 3: Duration = count of pair-years with dissimilarity > threshold (no consecutiveness)
threshold = 0.15

# 1 - similarity for each pair-year, sorted chronologically
inv_sorted = cosine_ic.sort('pair_end_year').map(
    lambda img: ee.Image(1).subtract(ee.Image(img)).rename('inv')
        .set('pair_end_year', img.get('pair_end_year'))
        .clip(austin)
)

# Boolean mask where dissimilarity > threshold
mask_ic = inv_sorted.map(
    lambda img: ee.Image(img).select('inv').gt(threshold).rename('mask').toInt16()
)

# Count of years above threshold (0..len(years)-1)
duration_of_changes = mask_ic.sum().toInt16().rename('duration')

# Combine layers into a single image
change_detection_image = (year_of_lowest_similarity
                          .addBands(magnitude_of_changes)
                          .addBands(duration_of_changes))

# ...existing code...

# # Visualization
# Map_change.addLayer(change_detection_image.select('year_of_lowest_similarity'), {
#     'min': 2017, 'max': 2024,
#     'palette': ['#2166ac','#4393c3','#92c5de','#fddbc7','#d6604d','#b2182b','#67001f']
# }, 'Year of Lowest Similarity')

# Map_change.addLayer(change_detection_image.select('magnitude_of_changes'), {
#     'min': 0, 'max': 1,
#     'palette': ['white','#fee08b','#f46d43','#a50026']
# }, 'Magnitude of Changes')

# Map_change.addLayer(change_detection_image.select('duration'), {
#     'min': 0, 'max': len(list(years)) - 1,   # number of consecutive pairs, e.g., 7 for 2017–2024
#     'palette': ['#ffffbf','#fdae61','#f46d43','#d73027','#a50026']
# }, 'Duration (count > 0.2)')


# Combine layers
change_detection_image = (year_of_lowest_similarity
                          .addBands(magnitude_of_changes)
                          .addBands(duration_of_changes))

# Visualization (single, no undefined vars)
MapAlpha_change.addLayer(
    change_detection_image.select('year_of_lowest_similarity'),
    {
        'min': 2017, 'max': 2024,
        'palette': ['#2166ac','#4393c3','#92c5de','#fddbc7','#d6604d','#b2182b','#67001f']
    },
    'Year of Lowest Similarity'
)

MapAlpha_change.addLayer(
    change_detection_image.select('magnitude_of_changes'),
    {
        'min': 0, 'max': 1,
        'palette': ['white','#fee08b','#f46d43','#a50026']
    },
    'Magnitude of Changes'
)

MapAlpha_change.addLayer(
    change_detection_image.select('duration'),
    {
        'min': 0, 'max': len(list(years)) - 1,  # 7 for 2017–2024
        'palette': ['#ffffbf','#fdae61','#f46d43','#d73027','#a50026']
    },
    'Duration (count > 0.15)'
)

MapAlpha_change



In [ ]:
# Smooth magnitude and highlight change hotspots (>= 0.1), removing small noisy blobs
kernel = ee.Kernel.gaussian(radius=2, sigma=1, units='pixels')

magnitude = change_detection_image.select('magnitude_of_changes')
magnitude_smoothed = magnitude.convolve(kernel).rename('magnitude_smoothed')

threshold = 0.1
hot_mask = magnitude_smoothed.gte(threshold)

# Remove small clusters using connected pixel count (8-neighborhood)
min_cluster_pixels = 25  # adjust as needed
cluster_sizes = hot_mask.connectedPixelCount(maxSize=8, eightConnected=True)
hot_clusters = hot_mask.updateMask(cluster_sizes.gte(min_cluster_pixels))

# Visualization
smoothed_vis = {
    'min': 0, 'max': 1,
    'palette': ['#ffffff', '#fdae61', '#f46d43', '#d73027', '#a50026']  # white -> red
}

# New map for smoothed magnitude + hotspots
MapAlpha_Smoothed = geemap.Map(center=[30.2672, -97.7431], zoom=10)
MapAlpha_Smoothed.addLayer(austin, {'color': 'blue'}, 'Austin')
MapAlpha_Smoothed.addLayer(magnitude, {'min': 0, 'max': 1, 'palette': ['white','#fee08b','#f46d43','#a50026']}, 'Magnitude (original)')
MapAlpha_Smoothed.addLayer(magnitude_smoothed, smoothed_vis, 'Magnitude (smoothed)')
MapAlpha_Smoothed.addLayer(hot_clusters.selfMask(), {'palette': ['#ff0000']}, 'Hotspots (>= 0.1, clustered)')
MapAlpha_Smoothed


In [ ]:
# =============================================================================
# Changes-only map: keep YOD/MAG/DUR only where smoothed MAG >= 0.1
# =============================================================================
kernel = ee.Kernel.gaussian(radius=2, sigma=1, units='pixels')

mag = change_detection_image.select('magnitude_of_changes')
mag_smoothed = mag.convolve(kernel).rename('mag_smoothed')

# Threshold and cluster filtering
mag_thresh = 0.1
raw_mask = mag_smoothed.gte(mag_thresh)

# Important: use a large maxSize so clusters can exceed min_cluster_pixels
min_cluster_pixels = 25
cluster_sizes = raw_mask.connectedPixelCount(maxSize=1024, eightConnected=True)
change_mask = raw_mask.updateMask(cluster_sizes.gte(min_cluster_pixels)).selfMask()

# Build final 3-band labels (replace MAG with smoothed MAG) and mask to changes
final_labels = ee.Image.cat([
    change_detection_image.select('year_of_lowest_similarity').rename('yod'),
    mag_smoothed.rename('mag'),
    change_detection_image.select('duration').rename('dur')
]).updateMask(change_mask)

# Map
MapAlpha_ChangesOnly = geemap.Map(center=[30.2672, -97.7431], zoom=10)
MapAlpha_ChangesOnly.addLayer(austin, {'color': 'blue'}, 'Austin')

# Optional hotspot overlay
MapAlpha_ChangesOnly.addLayer(change_mask, {'palette': ['#ff0000']}, 'Hotspots (>= 0.1, clustered)', True)

# MAG (smoothed) in changed areas
MapAlpha_ChangesOnly.addLayer(
    final_labels.select('mag'),
    {'min': 0, 'max': 1, 'palette': ['#ffffff', '#fdae61', '#f46d43', '#d73027', '#a50026']},
    'MAG (smoothed, changes only)', False
)

# YOD in changed areas
MapAlpha_ChangesOnly.addLayer(
    final_labels.select('yod'),
    {'min': 2017, 'max': 2024,
     'palette': ['#2166ac','#4393c3','#92c5de','#fddbc7','#d6604d','#b2182b','#67001f']},
    'YOD (changes only)', False
)

# DUR in changed areas
MapAlpha_ChangesOnly.addLayer(
    final_labels.select('dur'),
    {'min': 0, 'max': len(list(years)) - 1,
     'palette': ['#ffffbf','#fdae61','#f46d43','#d73027','#a50026']},
    'DUR (changes only)', False
)

# Expose for reuse
globals()['change_mask'] = change_mask
globals()['final_labels'] = final_labels
globals()['mag_smoothed'] = mag_smoothed

MapAlpha_ChangesOnly